# イジング模型シミュレーション：コードの一行解説

このノートでは、シミュレーションの核となるプログラムを、一行ずつ何をしているのか詳しく解説します。

---

## 1. イジング模型クラスの定義

まずは、磁石（スピン）の状態とルールを管理する「クラス」の解説です。

In [ ]:
class IsingModel:
    def __init__(self, L, T, J=1.0):
        self.L = L            # 1辺の長さ
        self.T = T            # 温度
        self.J = J            # 相互作用の強さ (1.0 = 強磁性)
        self.N = L * L        # 全サイト数 (面積)
        # ランダムに +1 または -1 で格子を埋める
        self.spins = np.random.choice([1, -1], size=(L, L))

### 解説：
- `__init__`: オブジェクトを作った時に最初に呼ばれる設定です。ここでサイズや温度を決めています。
- `np.random.choice([1, -1], size=(L, L))`: これが「スピンの初期状態」です。真っ白なキャンバスにランダムに $+1$ と $-1$ を塗るイメージです。

---

## 2. モンテカルロ・ステップ (1 MCS)

ここがこのプログラムの「心臓部」です。全スピンを更新しようと試みます。

In [ ]:
    def mcs_step(self):
        # N回（全サイト分）の更新を試みるので O(L^2) になる
        for _ in range(self.N):
            # 1. どこを更新するかランダムに選ぶ
            i = np.random.randint(0, self.L)
            j = np.random.randint(0, self.L)
            
            S = self.spins[i, j]  # 選んだ場所の現在のスピン (+1 or -1)
            
            # 2. 周囲4つのスピンの合計を計算 (周期境界条件)
            neighbor_sum = (
                self.spins[(i+1)%self.L, j] + 
                self.spins[(i-1)%self.L, j] + 
                self.spins[i, (j+1)%self.L] + 
                self.spins[i, (j-1)%self.L]
            )
            
            # 3. 反転させた場合のエネルギー変化を計算
            dE = 2 * self.J * S * neighbor_sum
            
            # 4. メトロポリス判定：ひっくり返すかどうか決める
            if dE <= 0 or np.random.rand() < np.exp(-dE / self.T):
                self.spins[i, j] *= -1  # 反転 (+1 -> -1, -1 -> +1)

### 解説：
- `(i+1)%self.L`: これが**「周期境界条件」**です。右端を超えたら左端に繋がるようにしています（ドーナツ型）。
- `dE = 2 * J * S * sum`: スピン $S$ を $-S$ に変えると、エネルギーは「$-J \cdot S \cdot sum$」から「$+J \cdot S \cdot sum$」に変わります。その差は $2J S \cdot sum$ です。
- `np.random.rand() < np.exp(-dE / T)`: 
    - `dE <= 0` ならエネルギーが下がるので必ずひっくり返します。
    - `dE > 0` なら「熱ゆらぎ」に賭けます。温度 $T$ が高いほど `exp(-dE/T)` は $1$ に近くなり、ひっくり返りやすくなります。

---

## 3. シミュレーションの実行フロー

最後に、このクラスを使ってどうやって結果を出すかの流れです。

In [ ]:
def run_simulation(L, T):
    model = IsingModel(L, T)     # 模型を用意
    
    # --- 焼きなまし (Burn-in) ---
    # 最初はランダムすぎるので、しばらく動かして落ち着かせる
    for _ in range(500): model.mcs_step()
    
    ms = []
    # --- 本番測定 ---
    for _ in range(2000):
        model.mcs_step()         # 1回動かす
        ms.append(np.mean(model.spins))  # その時の磁化を記録
    
    return np.array(ms)

### 解説：
- **焼きなまし (Burn-in)**: 準備体操のようなものです。初期状態がデタラメなので、本番データを取る前にその温度の「ふさわしい状態」まで慣らしています。
- `ms.append(np.mean(model.spins))`: 刻一刻と変化する磁石の様子を、リストに溜め込んでいきます。これが後で「ビンダー累積量」の計算に使われます。